# 07 — Geocode Priority Neighborhoods & Generate Policy Brief

1. Load gap surface + recompute priority scores
2. Download Census TIGER place boundaries (WA)
3. Spatial join BG centroids → named places (fallback: county subdivision)
4. Build named priority table
5. Write `data/outputs/policy_brief.md`

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import geopandas as gpd
import pandas as pd
import requests, zipfile, io
from shapely.geometry import Point

/home/simonhans/anaconda3/lib/python3.7/site-packages/geopandas/_compat.py:115: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  shapely_geos_version, geos_capi_version_string


In [2]:
# Load gap surface
gap_gdf = gpd.read_file('../data/processed/gap_surface.gpkg')

# Recompute normalized scores and equity distance
v = gap_gdf['vulnerability_index']
a = gap_gdf['norm_accessibility']
vuln_n = (v - v.min()) / (v.max() - v.min())
acc_n  = (a - a.min()) / (a.max() - a.min())
gap_gdf['equity_distance'] = (vuln_n - acc_n) / np.sqrt(2)

under = gap_gdf[gap_gdf['equity_distance'] > 0].copy()
under['gap_score_n']   = (under['gap_score']       - under['gap_score'].min())       / (under['gap_score'].max()       - under['gap_score'].min())
under['equity_dist_n'] = (under['equity_distance'] - under['equity_distance'].min()) / (under['equity_distance'].max() - under['equity_distance'].min())
under['priority']      = (under['gap_score_n'] + under['equity_dist_n']) / 2

print(f"{len(gap_gdf)} block groups total, {len(under)} underserved")

575 block groups total, 555 underserved


In [3]:
# Download Census TIGER place + county subdivision boundaries for WA (state FIPS 53)
TIGER_BASE = "https://www2.census.gov/geo/tiger/TIGER2023"
CACHE = Path('../data/raw/tiger')
CACHE.mkdir(parents=True, exist_ok=True)

def fetch_tiger(layer, filename):
    out = CACHE / filename
    if out.exists():
        print(f"  cached: {filename}")
        return gpd.read_file(out)
    url = f"{TIGER_BASE}/{layer}/{filename}"
    print(f"  downloading {url}")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall(CACHE)
    shp = next(CACHE.glob(filename.replace('.zip', '.shp')))
    return gpd.read_file(shp)

places = fetch_tiger('PLACE',  'tl_2023_53_place.zip')
cousub = fetch_tiger('COUSUB', 'tl_2023_53_cousub.zip')

# Filter cousub to Snohomish County (FIPS 061)
cousub = cousub[cousub['COUNTYFP'] == '061'].copy()

print(f"Places: {len(places)}, County subdivisions (Snohomish): {len(cousub)}")

  downloading https://www2.census.gov/geo/tiger/TIGER2023/PLACE/tl_2023_53_place.zip
  downloading https://www2.census.gov/geo/tiger/TIGER2023/COUSUB/tl_2023_53_cousub.zip
Places: 639, County subdivisions (Snohomish): 13


In [4]:
# Build block group centroids in WGS84
bg_wgs = gap_gdf[['GEOID', 'geometry']].copy().to_crs('EPSG:4326')
bg_wgs['centroid'] = bg_wgs.geometry.centroid
centroids = gpd.GeoDataFrame(
    bg_wgs[['GEOID']], geometry=bg_wgs['centroid'], crs='EPSG:4326'
)

# Reproject TIGER to match
places_wgs = places[['NAME', 'geometry']].to_crs('EPSG:4326')
cousub_wgs = cousub[['NAME', 'geometry']].to_crs('EPSG:4326').rename(columns={'NAME': 'COUSUB_NAME'})

# Join 1: named places (cities, CDPs)
joined = gpd.sjoin(centroids, places_wgs, how='left', predicate='within')
joined = joined[['GEOID', 'NAME']].rename(columns={'NAME': 'place_name'})

# Join 2: county subdivision as fallback for unmatched
unmatched = joined[joined['place_name'].isna()][['GEOID']].copy()
unmatched_pts = centroids[centroids['GEOID'].isin(unmatched['GEOID'])]
fallback = gpd.sjoin(unmatched_pts, cousub_wgs, how='left', predicate='within')
fallback = fallback[['GEOID', 'COUSUB_NAME']].rename(columns={'COUSUB_NAME': 'place_name'})

# Merge fallback names in
joined = joined.set_index('GEOID')
joined.update(fallback.set_index('GEOID'))
joined = joined.reset_index()

# Still unmatched → label as county unincorporated
joined['place_name'] = joined['place_name'].fillna('Snohomish County (unincorporated)')

gap_gdf = gap_gdf.merge(joined, on='GEOID', how='left')
under   = under.merge(joined, on='GEOID', how='left')

print("Sample place names:")
print(gap_gdf[['GEOID', 'place_name']].dropna().head(10).to_string(index=False))

/home/simonhans/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  This is separate from the ipykernel package so we can avoid doing imports until


Sample place names:
       GEOID          place_name
530610527084          Marysville
530619400014 Tulalip Reservation
530610531021            Stanwood
530610531023        Lake Goodwin
530610533025            Stanwood
530610534003            Stanwood
530610534004            Stanwood
530610538012              Sultan
530610538031             Startup
530610401002             Everett


In [5]:
import time, requests as req

def parse_tract(geoid):
    """Extract human-readable tract + BG from GEOID.
    GEOID = SS CCC TTTTTT B  (2+3+6+1)
    Tract field is 6 digits: first 4 = integer, last 2 = decimal.
    e.g. '041814' → Tract 418.14, but if last 2 are '00' → Tract 4181
    """
    tract_raw = geoid[5:11]
    bg = geoid[11]
    major, minor = tract_raw[:4].lstrip('0') or '0', tract_raw[4:]
    tract_str = f"{major}.{minor}" if minor != '00' else major
    return f"Tract {tract_str}, BG {bg}"

def nominatim_subname(lat, lon, retries=2):
    """Return the most specific sub-city name from Nominatim, or None."""
    url = 'https://nominatim.openstreetmap.org/reverse'
    params = {'format': 'json', 'lat': lat, 'lon': lon, 'zoom': 16, 'addressdetails': 1}
    headers = {'User-Agent': 'snotrac-transit-equity-analysis/1.0'}
    for _ in range(retries):
        try:
            r = req.get(url, params=params, headers=headers, timeout=10)
            r.raise_for_status()
            addr = r.json().get('address', {})
            for key in ('neighbourhood', 'suburb', 'quarter', 'hamlet', 'village'):
                if key in addr:
                    return addr[key]
            return None
        except Exception:
            time.sleep(2)
    return None

# Only geocode top 15 priority BGs — stays within Nominatim rate limit (~1 req/s)
under_tmp = gap_gdf[gap_gdf['equity_distance'] > 0].copy()
under_tmp['gap_score_n']   = (under_tmp['gap_score'] - under_tmp['gap_score'].min()) / \
                               (under_tmp['gap_score'].max() - under_tmp['gap_score'].min())
under_tmp['equity_dist_n'] = (under_tmp['equity_distance'] - under_tmp['equity_distance'].min()) / \
                               (under_tmp['equity_distance'].max() - under_tmp['equity_distance'].min())
under_tmp['priority']      = (under_tmp['gap_score_n'] + under_tmp['equity_dist_n']) / 2
top15_idx = under_tmp.nlargest(15, 'priority').index

# Compute centroids in WGS84
top15_geo = gap_gdf.loc[top15_idx, ['GEOID', 'place_name', 'geometry']].copy().to_crs('EPSG:4326')
top15_geo['centroid'] = top15_geo.geometry.centroid

print('Reverse geocoding top 15 priority block groups...')
subnames = {}
for _, row in top15_geo.iterrows():
    lat, lon = row['centroid'].y, row['centroid'].x
    sub = nominatim_subname(lat, lon)
    city = row['place_name']
    tract = parse_tract(row['GEOID'])
    subnames[row['GEOID']] = f"{city} — {sub}" if sub and sub.lower() != city.lower() \
                              else f"{city} — {tract}"
    print(f"  {row['GEOID']}  {subnames[row['GEOID']]}")
    time.sleep(1.1)  # Nominatim ToS: max 1 req/s

gap_gdf['display_name'] = gap_gdf.apply(
    lambda r: subnames.get(r['GEOID'], r['place_name']), axis=1
)

Reverse geocoding top 15 priority block groups...
  530610418143  Everett — Twin Creeks


/home/simonhans/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:44: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



  530610419044  Everett — Casino Corner
  530610533013  Stanwood — North Stanwood
  530610416014  Everett — Valley View
  530610537001  Darrington — Tract 537, BG 1
  530610402004  Everett — Delta
  530610535064  Arlington Heights — Tract 535.06, BG 4
  530610538013  Sultan — Tract 538.01, BG 3
  530619400021  Tulalip Reservation — Tulalip Bay
  530610533025  Stanwood — Tract 533.02, BG 5
  530610533022  Stanwood — Tract 533.02, BG 2
  530619400012  Tulalip Reservation — Shaker Church
  530610402005  Everett — Delta
  530610419043  Everett — Casino Corner
  530610535051  Arlington — Tract 535.05, BG 1


In [6]:
# Named priority table
# Bring display_name (set by nominatim cell) into the underserved subset
under = under.merge(gap_gdf[['GEOID', 'display_name']], on='GEOID', how='left')

top15 = under.nlargest(15, 'priority')[[
    'GEOID', 'display_name', 'vulnerability_index',
    'norm_accessibility', 'gap_score', 'equity_distance', 'priority'
]].reset_index(drop=True)
top15.index += 1

for col in ['vulnerability_index', 'norm_accessibility', 'gap_score', 'equity_distance', 'priority']:
    top15[col] = top15[col].round(3)

print(top15.to_string())

           GEOID                            display_name  vulnerability_index  norm_accessibility  gap_score  equity_distance  priority
1   530610418143                   Everett — Twin Creeks                0.443               0.003      0.354            0.531     0.907
2   530610419044                 Everett — Casino Corner                0.570               0.047      0.256            0.653     0.855
3   530610533013               Stanwood — North Stanwood                0.352               0.000      0.352            0.424     0.821
4   530610416014                   Everett — Valley View                0.340               0.000      0.340            0.409     0.792
5   530610537001            Darrington — Tract 537, BG 1                0.335               0.000      0.335            0.404     0.781
6   530610402004                         Everett — Delta                0.527               0.065      0.212            0.590     0.743
7   530610535064  Arlington Heights — Tract 535.

In [7]:
from datetime import date
import tabulate
# --- summary stats for the brief ---
n_bg          = len(gap_gdf)
n_underserved = int((gap_gdf['equity_distance'] > 0).sum())
pct_under     = round(100 * n_underserved / n_bg)
top_places    = top15['display_name'].value_counts().head(5)

# Build markdown table of top 10 for the brief
brief_table = top15.head(10)[['display_name', 'vulnerability_index', 'norm_accessibility', 'gap_score']].copy()
brief_table.columns = ['Neighborhood', 'Vulnerability', 'Accessibility', 'Gap score']
brief_table.index.name = 'Rank'
md_table = brief_table.to_markdown()

brief = f"""# Snohomish County Transit Equity Gap
**A data brief for Community Transit and Snotrac — {date.today().strftime('%B %Y')}**

---

## The question

Are the residents who need public transit most — elderly, disabled, carless, and low-income —
actually living near it? This analysis uses Community Transit's own published schedule and
U.S. Census data to answer that question for all {n_bg} block groups in Snohomish County.

---

## What we found

- **{pct_under}% of Snohomish County block groups** ({n_underserved} of {n_bg}) fall below
  the equity line — they have more vulnerable residents than their transit service would justify.
- The gap is not random. It is concentrated in specific, nameable communities — most of which
  already have thin or no service.
- A small number of block groups sit above the equity line: better-served than their population
  need would predict. These highlight where past investment has been concentrated.

---

## How the gap is measured

- **Vulnerability index** — a composite of elderly population, disability rate, zero-vehicle
  households, and households below 200% of the poverty line (Census ACS 2023).
- **Accessibility score** — frequency-weighted, network-distance access to every bus stop
  in the county (Community Transit GTFS, March 2026). A stop served every 15 minutes counts
  for more than one served twice a day.
- **Equity line** — a diagonal from (0,0) to (1,1) on the vulnerability–accessibility plane.
  Points below it are underserved: high need, low service. **Distance below the line is the gap magnitude.**

![Vulnerability vs. Accessibility — all 575 Snohomish County block groups. Points below the diagonal are underserved.](vulnerability_accessibility.png)

---

## Priority neighborhoods

The 10 highest-priority block groups, ranked by a combined score of absolute need
and equity misalignment:

{md_table}

*Vulnerability and accessibility are normalized 0–1 across the county.
Gap score = vulnerability × (1 − accessibility). Full ranked list available on request.*

![Priority neighborhoods mapped across Snohomish County. Brighter fill = higher combined priority score.](priority_map.png)

---

## The ask

These 10 neighborhoods represent the highest return on investment for any new service,
frequency increase, or demand-response expansion in the county.
We are offering this analysis — and the underlying data — as a free resource to support
Snotrac's advocacy work and Community Transit's service planning process.

**We would like 20 minutes to walk through the findings with your team.**

---

*Data: U.S. Census Bureau ACS 5-Year Estimates (2023), Community Transit GTFS (March 2026),
OpenStreetMap. Analysis methodology available on request.*
"""

out = Path('../data/outputs/policy_brief.md')
out.write_text(brief)
print(f"Brief written → {out}")
print()
print(brief)

Brief written → ../data/outputs/policy_brief.md

# Snohomish County Transit Equity Gap
**A data brief for Community Transit and Snotrac — April 2026**

---

## The question

Are the residents who need public transit most — elderly, disabled, carless, and low-income —
actually living near it? This analysis uses Community Transit's own published schedule and
U.S. Census data to answer that question for all 575 block groups in Snohomish County.

---

## What we found

- **97% of Snohomish County block groups** (555 of 575) fall below
  the equity line — they have more vulnerable residents than their transit service would justify.
- The gap is not random. It is concentrated in specific, nameable communities — most of which
  already have thin or no service.
- A small number of block groups sit above the equity line: better-served than their population
  need would predict. These highlight where past investment has been concentrated.

---

## How the gap is measured

- **Vulnerability index